## Configuration


In [ ]:

class Config():
    def __init__(self):
        self.db_name = "workout"
        self.max_FilesPerTrigger = 100

In [3]:
config = Config()
print(f"Database Name: {config.db_name}")

Database Name: workout


## Set Up


In [2]:
import os
def get_spark():
    """Get Spark session based on environment"""
    
    if "DATABRICKS_RUNTIME_VERSION" in os.environ:
        print("Running in Databricks environment")
        from databricks.sdk.runtime import spark
        return spark
    else:
        print("Running locally with Databricks Connect")
        from databricks.connect import DatabricksSession
        spark = (DatabricksSession
                .builder
                .profile("dev-free-edition")
                .serverless(True)
                .getOrCreate())
        return spark


spark = get_spark()
spark

Running locally with Databricks Connect


In [ ]:

class SetUp():
    def __init__(self,env):
        conf = Config()
        self.db_name = conf.db_name
        self.catalog = env
        self.initialized = False


    def create_db(self):
        print(f"Creating the database {self.catalog}.{self.db_name}...", end='')
        spark.sql(f"CREATE DATABASE IF NOT EXISTS {self.catalog}.{self.db_name}")
        spark.sql(f"USE {self.catalog}.{self.db_name}")
        self.initialized = True
        print("Done.")













In [10]:
setup = SetUp(env="dev")


In [ ]:
setup.create_db()

In [9]:
%sql

show databases in dev;

,databaseName
0,default
1,information_schema
2,rp
3,rw
4,sw
5,test
6,workout


In [1]:

def get_secrets(name: str,  scope: str = "test", env: str = "local"):
    """Get secrets from environment variables or Databricks Secrets"""

    if env == "local":

        from dotenv import load_dotenv
        import os

        load_dotenv()
        return os.getenv(name)
    
    try:
        return dbutils.secrets.get(scope=scope, key=name)
    except Exception as e:
        print(f"Error getting secret {name} from scope {scope}: {e}")

    raise ValueError(f"Secret {name} not found in scope {scope}")


In [8]:
azure_storage_account = get_secrets("azure_storage_account")


In [3]:
catalog = "dev"

ingestion_db = "control"
ingestion_table = "ingest_table_registry"
bronze_db = "bronze_workout"


# spark.sql(f"CREATE DATABASE IF NOT EXISTS {catalog}.{ingestion_db};")

# spark.sql(f"CREATE DATABASE IF NOT EXISTS {catalog}.{bronze_db};")

In [3]:
query = f"""

CREATE TABLE IF NOT EXISTS {catalog}.{ingestion_db}.{ingestion_table} (
  table_id BIGINT GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1),
  source_path STRING NOT NULL,
  source_format STRING DEFAULT 'parquet',
  header BOOLEAN DEFAULT TRUE,
  table_version INT DEFAULT 1,

  target_catalog STRING DEFAULT 'dev',
  target_schema STRING DEFAULT 'test',
  target_table STRING NOT NULL,
  write_mode STRING DEFAULT 'append'
      CHECK (write_mode IN ('append', 'merge', 'overwrite')),
  checkpoint_path STRING NOT NULL,
  merge_key ARRAY<STRING> DEFAULT ARRAY(),
  partition_by ARRAY<STRING> DEFAULT ARRAY(),
  enabled BOOLEAN DEFAULT TRUE,

 trigger_mode STRING DEFAULT 'availableNow'
    CHECK (trigger_mode IN ('availableNow', 'processingTime', 'continuous')),
  processing_time STRING DEFAULT '5 minutes',
  schema_evolution_mode STRING DEFAULT 'BACKWARD'
    CHECK (schema_evolution_mode IN ('BACKWARD', 'FORWARD', 'FULL', 'NONE')),
  
  expected_rows_per_day BIGINT,
  last_success TIMESTAMP,
  last_error STRING,
  consecutive_failures INT,
  last_run_duration_seconds INT,
  created_by STRING,
  created_at TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP(),
  updated_by STRING,
  updated_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP()
)
TBLPROPERTIES (
  'delta.enableChangeDataFeed' = 'true',
  'delta.autoOptimize.autoCompact' = 'true',
  'delta.feature.allowColumnDefaults' = 'supported'
);

"""

In [17]:
spark.sql(query)

""


In [ ]:
insert_query = f"""

INSERT INTO {catalog}.{ingestion_db}.{ingestion_table} 
(source_path,                         source_format,header,table_version,target_catalog,target_schema,target_table,write_mode,checkpoint_path,merge_key,partition_by,enabled,trigger_mode,processing_time,schema_evolution_mode,expected_rows_per_day,last_success,last_error,consecutive_failures,last_run_duration_seconds,created_by,created_at,updated_by,updated_at)
VALUES 
('/Volumes/dev/bronze_workout/rw/bpm/', 'json', TRUE, 1, 'dev', 'bronze_workout', 'bpm', 'append', '/Volumes/dev/bronze_workout/checkpoints/', ARRAY(), ARRAY(), TRUE, 'availableNow', '5 minutes', 'BACKWARD', 1000000, NULL, NULL, 0, NULL, 'gaurav.thagunna', CURRENT_TIMESTAMP(), NULL, CURRENT_TIMESTAMP());

"""

spark.sql(insert_query)

,num_affected_rows,num_inserted_rows
0,1,1


In [4]:
configs = spark.sql(f"select * from {catalog}.{ingestion_db}.{ingestion_table}")
configs.show(truncate=False)

+--------+-------------------------------------------+-------------+------+-------------+--------------+--------------+------------+----------+----------------------------------------+---------+------------+-------+------------+---------------+---------------------+---------------------+------------+----------+--------------------+-------------------------+---------------+--------------------------+----------+--------------------------+
|table_id|source_path                                |source_format|header|table_version|target_catalog|target_schema |target_table|write_mode|checkpoint_path                         |merge_key|partition_by|enabled|trigger_mode|processing_time|schema_evolution_mode|expected_rows_per_day|last_success|last_error|consecutive_failures|last_run_duration_seconds|created_by     |created_at                |updated_by|updated_at                |
+--------+-------------------------------------------+-------------+------+-------------+--------------+--------------

In [5]:
configs_dict = configs.select("table_id","source_path","source_format","header","target_catalog","target_schema","table_version","target_table","write_mode","checkpoint_path","merge_key","partition_by","enabled","trigger_mode","processing_time","schema_evolution_mode").collect()[0].asDict()
configs_dict

{'table_id': 1,
 'source_path': '/Volumes/dev/bronze_workout/rw/dataset/bpm/',
 'source_format': 'json',
 'header': True,
 'target_catalog': 'dev',
 'target_schema': 'bronze_workout',
 'table_version': 1,
 'target_table': 'bpm',
 'write_mode': 'append',
 'checkpoint_path': '/Volumes/dev/bronze_workout/checkpoints/',
 'merge_key': [],
 'partition_by': [],
 'enabled': True,
 'trigger_mode': 'availableNow',
 'processing_time': '5 minutes',
 'schema_evolution_mode': 'BACKWARD'}

In [6]:
def get_schema_file(path:str):
    df = spark.read.json(path)
    return df.schema


In [7]:
schema = get_schema_file(path=configs_dict["source_path"])
schema

StructType([StructField('key', LongType(), True), StructField('offset', LongType(), True), StructField('partition', LongType(), True), StructField('timestamp', LongType(), True), StructField('topic', StringType(), True), StructField('value', StructType([StructField('device_id', LongType(), True), StructField('heartrate', DoubleType(), True), StructField('time', LongType(), True)]), True)])

In [9]:
ingestion_db = "control"
schema_registry_tb = "schema_registry"

schema_registry_query = f"""CREATE TABLE IF NOT EXISTS {catalog}.{ingestion_db}.{schema_registry_tb} (
  id BIGINT GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1),
  table_id INT NOT NULL,
  table_name STRING NOT NULL,
  version INT DEFAULT 1,
  is_latest BOOLEAN DEFAULT TRUE,                 
  schema_format STRING,          
  schema_definition STRING,     
  created_by STRING,
  created_at TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP()
  )
  TBLPROPERTIES (
    'delta.autoOptimize.autoCompact' = 'true',
    'delta.feature.allowColumnDefaults' = 'supported'
  );

  """

spark.sql(schema_registry_query)

""


In [11]:
spark.table(f"{catalog}.{ingestion_db}.{schema_registry_tb}").show()

+---+--------+----------+-------+---------+-------------+-----------------+----------+----------+
| id|table_id|table_name|version|is_latest|schema_format|schema_definition|created_by|created_at|
+---+--------+----------+-------+---------+-------------+-----------------+----------+----------+
+---+--------+----------+-------+---------+-------------+-----------------+----------+----------+



In [22]:
spark.sql(f"SELECT * FROM {catalog}.{ingestion_db}.{schema_registry_tb}").show(truncate=False)

+---+--------+----------+-------+---------+-------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+---------------+--------------------------+
|id |table_id|table_name|version|is_latest|schema_format|schema_definition                                                                                                                                                                                                               

In [20]:
query = f"""

INSERT INTO {catalog}.{ingestion_db}.{schema_registry_tb} (
  table_id,
  table_name,
  version,
  is_latest,
  schema_format,
  schema_definition,
  created_by
)
VALUES (
  {configs_dict["table_id"]},
  '{configs_dict["target_table"]}',
  1,
  TRUE,
  'json',
  '{schema.json()}',
  'gaurav.thagunna'
)

"""

spark.sql(query)



,num_affected_rows,num_inserted_rows
0,1,1


In [24]:
spark.table("dev.control.schema_registry").show(truncate=False)

+---+--------+----------+-------+---------+-------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+---------------+--------------------------+
|id |table_id|table_name|version|is_latest|schema_format|schema_definition                                                                                                                                                                                                               

In [7]:
schema_row = (
    spark.table("dev.control.schema_registry")
         .filter("table_id = 1 AND is_latest = TRUE").limit(1).collect()[0]


)
schema_row

Row(id=1, table_id=1, table_name='bpm', version=1, is_latest=True, schema_format='json', schema_definition='{"fields":[{"metadata":{},"name":"key","nullable":true,"type":"long"},{"metadata":{},"name":"offset","nullable":true,"type":"long"},{"metadata":{},"name":"partition","nullable":true,"type":"long"},{"metadata":{},"name":"timestamp","nullable":true,"type":"long"},{"metadata":{},"name":"topic","nullable":true,"type":"string"},{"metadata":{},"name":"value","nullable":true,"type":{"fields":[{"metadata":{},"name":"device_id","nullable":true,"type":"long"},{"metadata":{},"name":"heartrate","nullable":true,"type":"double"},{"metadata":{},"name":"time","nullable":true,"type":"long"}],"type":"struct"}}],"type":"struct"}', created_by='gaurav.thagunna', created_at=datetime.datetime(2026, 1, 27, 18, 39, 10, 25041))

In [8]:
from pyspark.sql.types import StructType
import json
schema = StructType.fromJson(json.loads(schema_row.schema_definition))
schema

StructType([StructField('key', LongType(), True), StructField('offset', LongType(), True), StructField('partition', LongType(), True), StructField('timestamp', LongType(), True), StructField('topic', StringType(), True), StructField('value', StructType([StructField('device_id', LongType(), True), StructField('heartrate', DoubleType(), True), StructField('time', LongType(), True)]), True)])

In [ ]:
spark.read.json(configs_dict["source_path"]).show(truncate=False)

+------+------+---------+----------+-----+----------------------------------------+
|key   |offset|partition|timestamp |topic|value                                   |
+------+------+---------+----------+-----+----------------------------------------+
|118440|0     |0        |1678410000|bpm  |{118440, 41.21897868284929, 1678410000} |
|118440|1     |1        |1678410001|bpm  |{118440, 46.95949313912609, 1678410001} |
|118440|2     |2        |1678410002|bpm  |{118440, 84.39302961093657, 1678410002} |
|118440|3     |3        |1678410003|bpm  |{118440, 87.4902278952101, 1678410003}  |
|118440|4     |4        |1678410004|bpm  |{118440, 76.16501196545016, 1678410004} |
|118440|5     |5        |1678410005|bpm  |{118440, 42.46430112225653, 1678410005} |
|118440|6     |6        |1678410006|bpm  |{118440, 46.989738006342094, 1678410006}|
|118440|7     |7        |1678410007|bpm  |{118440, 93.72495479825955, 1678410007} |
|118440|8     |8        |1678410008|bpm  |{118440, 85.67159141451009, 167841

In [9]:

df = spark.read.schema(schema).json(configs_dict["source_path"])
df.show(truncate=False)

+------+------+---------+----------+-----+----------------------------------------+
|key   |offset|partition|timestamp |topic|value                                   |
+------+------+---------+----------+-----+----------------------------------------+
|118440|0     |0        |1678410000|bpm  |{118440, 41.21897868284929, 1678410000} |
|118440|1     |1        |1678410001|bpm  |{118440, 46.95949313912609, 1678410001} |
|118440|2     |2        |1678410002|bpm  |{118440, 84.39302961093657, 1678410002} |
|118440|3     |3        |1678410003|bpm  |{118440, 87.4902278952101, 1678410003}  |
|118440|4     |4        |1678410004|bpm  |{118440, 76.16501196545016, 1678410004} |
|118440|5     |5        |1678410005|bpm  |{118440, 42.46430112225653, 1678410005} |
|118440|6     |6        |1678410006|bpm  |{118440, 46.989738006342094, 1678410006}|
|118440|7     |7        |1678410007|bpm  |{118440, 93.72495479825955, 1678410007} |
|118440|8     |8        |1678410008|bpm  |{118440, 85.67159141451009, 167841

In [12]:
df.printSchema()

root
 |-- key: long (nullable = true)
 |-- offset: long (nullable = true)
 |-- partition: long (nullable = true)
 |-- timestamp: long (nullable = true)
 |-- topic: string (nullable = true)
 |-- value: struct (nullable = true)
 |    |-- device_id: long (nullable = true)
 |    |-- heartrate: double (nullable = true)
 |    |-- time: long (nullable = true)



In [45]:
configs_dict

{'table_id': 1,
 'source_path': '/Volumes/dev/bronze_workout/rw/dataset/bpm/',
 'source_format': 'json',
 'header': True,
 'target_catalog': 'dev',
 'target_schema': 'bronze_workout',
 'table_version': 1,
 'target_table': 'bpm',
 'write_mode': 'append',
 'checkpoint_path': '/Volumes/dev/bronze_workout/checkpoints/',
 'merge_key': [],
 'partition_by': [],
 'enabled': True,
 'trigger_mode': 'availableNow',
 'processing_time': '5 minutes',
 'schema_evolution_mode': 'BACKWARD'}

In [10]:
from pyspark.sql import SparkSession, DataFrame

def load_date(spark, configs_dict,schema) -> DataFrame:
    """Load data from a given path with the provided schema."""

    return (spark.readStream
            .format("cloudFiles")
            .option("cloudFiles.format", configs_dict["source_format"])
            .option("cloudFiles.schemaLocation", configs_dict["checkpoint_path"]+"schema/"+configs_dict["target_table"])
            .option("cloudFiles.schemaEvolutionMode","rescue")
            .option("cloudFiles.maxFilesPerTrigger", 1)
            .schema(schema)
            .load(configs_dict["source_path"])

    )


In [11]:
raw_data = load_date(spark=spark,configs_dict=configs_dict,schema=schema)


In [12]:
display(raw_data)

AnalysisException: Queries with streaming sources must be executed with writeStream.start(), or from a streaming table or flow definition within a Lakeflow Declarative Pipeline.;
cloudFiles

JVM stacktrace:
org.apache.spark.sql.catalyst.ExtendedAnalysisException
	at org.apache.spark.sql.catalyst.analysis.UnsupportedOperationChecker$.throwError(UnsupportedOperationChecker.scala:719)
	at org.apache.spark.sql.catalyst.analysis.UnsupportedOperationChecker$.$anonfun$checkForBatch$2(UnsupportedOperationChecker.scala:68)
	at org.apache.spark.sql.catalyst.analysis.UnsupportedOperationChecker$.$anonfun$checkForBatch$2$adapted(UnsupportedOperationChecker.scala:65)
	at org.apache.spark.sql.catalyst.trees.TreeNode.foreachUp(TreeNode.scala:325)
	at org.apache.spark.sql.catalyst.trees.TreeNode.$anonfun$foreachUp$1(TreeNode.scala:324)
	at org.apache.spark.sql.catalyst.trees.TreeNode.$anonfun$foreachUp$1$adapted(TreeNode.scala:324)
	at scala.collection.immutable.Vector.foreach(Vector.scala:2125)
	at o

DataFrame[key: bigint, offset: bigint, partition: bigint, timestamp: bigint, topic: string, value: struct<device_id:bigint,heartrate:double,time:bigint>, _rescued_data: string]

In [13]:
def view_streaming_df(df, num_rows=20, timeout=30):
    """View streaming DataFrame in console"""
    query = (df.writeStream
        .format("console")
        .option("checkpointLocation", f"{configs_dict['checkpoint_path']}tmp/{id(df)}")  # Unique checkpoint
        .option("numRows", num_rows)
        .option("truncate", False)
        .trigger(availableNow=True)
        .start()
    )
    
    query.awaitTermination(timeout=timeout)
    query.stop()
    
    return query

In [14]:
view_streaming_df(df=raw_data, num_rows=20, timeout=30)

In [15]:

# Write to in-memory table
query = (raw_data.writeStream
    .format("memory")
    .queryName("temp_stream_view")  # Name for the temp table
    .option("checkpointLocation", f"{configs_dict['checkpoint_path']}tmp/{id(raw_data)}")  # Unique checkpoint
    .trigger(availableNow=True)
    .start()
)

# Query the in-memory table
spark.sql("SELECT * FROM temp_stream_view LIMIT 10").show()


AnalysisException: This query does not support recovering from checkpoint location. Delete /Volumes/dev/bronze_workout/checkpoints/tmp/4955376272/offsets to start over.

JVM stacktrace:
org.apache.spark.sql.AnalysisException
	at org.apache.spark.sql.errors.QueryCompilationErrors$.recoverQueryFromCheckpointUnsupportedError(QueryCompilationErrors.scala:4518)
	at org.apache.spark.sql.execution.streaming.runtime.ResolveWriteToStream$.resolveCheckpointLocation(ResolveWriteToStream.scala:146)
	at org.apache.spark.sql.execution.streaming.runtime.ResolveWriteToStream$$anonfun$apply$1.applyOrElse(ResolveWriteToStream.scala:52)
	at org.apache.spark.sql.execution.streaming.runtime.ResolveWriteToStream$$anonfun$apply$1.applyOrElse(ResolveWriteToStream.scala:48)
	at org.apache.spark.sql.catalyst.plans.logical.AnalysisHelper.$anonfun$resolveOperatorsDownWithPruning$2(AnalysisHelper.scala:201)
	at org.apache.spark.sql.catalyst.trees.CurrentOrigin$.withOrigin(origin.scala:121)
	at org.apache.spark.sql

In [ ]:
table = ingest_table_registry

CREATE TABLE IF NOT EXISTS control.ingest_table_registry (
  id INT,
  source_path STRING,
  source_format STRING,
  header BOOLEAN,
  table_id STRING,
  table_version INT,
  merge_key STRING,
  partition_by STRING,
  target_catalog STRING,
  target_schema STRING,
  target_table STRING,
  write_mode STRING,             -- append|merge|overwrite
  checkpoint_path STRING,
  enabled BOOLEAN,
  trigger_mode STRING,           -- availableNow | processingTime
  processing_time STRING,        -- "5 seconds"
  sla_hours INT,
  priority INT,
  expected_rows_per_day BIGINT,
  schema_compatibility STRING,   -- BACKWARD|FORWARD|FULL|NONE
  last_success TIMESTAMP,
  last_error STRING,
  consecutive_failures INT,
  last_run_duration_seconds INT,
  created_by STRING,
  created_at TIMESTAMP,
  updated_by STRING,
  updated_at TIMESTAMP
)
USING DELTA;


In [ ]:
CREATE TABLE control.bronze_tables (
  table_name STRING,
  source_path STRING,
  format STRING,
  header BOOLEAN,
  schema_id STRING,
  merge_key STRING,
  partition_by STRING,
  enabled BOOLEAN,
  trigger_mode STRING,       
  processing_time STRING,  
  owner STRING,
  created_at TIMESTAMP,
  updated_at TIMESTAMP
)
USING DELTA;

""


In [18]:
!pwd

/Users/adi/Library/CloudStorage/OneDrive-UltraTendencyInternationalGmbH/Databricks/real_projects/WorkoutTime_Analytics_Databricks/notebooks
